# Hardware validation — motion safety changes (branch `bug-fix-2026-09-29`)

Checks on real hardware what the unit tests can only check against a fake wire:

| § | What | Moves? |
|---|---|---|
| 1 | Connect in safe_mode; Pi agent redeploys | no |
| 2 | Every motion path refuses under safe_mode; `lab.gantry.x` has no motion methods | no |
| 3 | Stop/brake commands get through the safe_mode gate; `set_safe_mode(True)` brakes | no |
| 4 | Fence check (exact, ribbon) | no |
| 5 | Enable motion — **explicit** | — |
| 6 | Non-blocking `move_to()`; second move refused | **yes** |
| 7 | `pause()` cancels a move in flight; motion refused until `resume()` | **yes** |
| 8 | **Jog** (timed, then open-ended) and how to stop it | **yes** |
| 9 | `move_to_unfenced()` passes a fence `move_to()` refuses | **yes** |
| 10 | `estop()` → motion refused → `rearm()` leaves safe_mode on | **yes** (halt) |
| 11 | Homing: travel-based timeout, non-blocking, pause cancels it *(optional)* | **yes** |
| 12 | Pi-agent rangefinder scan: pause mid-pass keeps partial data *(optional)* | **yes** |
| 13 | Summary, park, safe_mode back on | — |

## How to stop motion — read before §5

1. **The STOP cell** (§5, it only contains `lab.pause()`). `move_to()`, homing and jogs no longer
   block the kernel, so you can run it any time. Keep it on screen.
2. **A cell that is still running** (the `[*]` marker; e.g. the timed jog) is blocking the kernel:
   press **Interrupt** (⏹ / `I`,`I`). Every motion cell here catches the interrupt and stops the axis.
3. **From a terminal**, in the repo root: `touch ESTOP` (hard halt) or `touch PAUSE` (resumable).
   §1 starts the sentinel watcher. Delete the file before `rearm()`/`resume()`.
4. **Physical e-stop** — always the last word.

Before running anything that moves: everyone clear, `TEST_AXIS` has at least `3 × STEP_MM` of
clear travel in the **positive** direction from where it starts, and `JOG_SPEED` is slow.

In [ ]:
import logging, os, time, math
from pyprojroot import here
os.chdir(here())

from laguna import FlumeLab
from laguna.safety import SafetyState
from laguna.robot.macron import MotionHalted, MoveHandle
from laguna.robot.macron.connection import SnapMotionError
from laguna.robot.macron.fences import BoxFence, FenceViolation
from laguna.robot.motion_arbiter import MotionBusyError

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# ---- Parameters: check these against the rig before §5 ----
CONFIG = "config/example_config.yaml"
TEST_AXIS = "X"        # axis used for moves/jogs
STEP_MM = 20.0         # small move size, mm (positive direction)
SPEED = 20.0           # mm/s for test moves
SLOW_SPEED = 5.0       # mm/s for the move that gets paused (needs to still be moving at ~1 s)
JOG_SPEED = 5.0        # mm/s, sign = direction
JOG_SECONDS = 2.0      # timed jog duration
RUN_HOMING = False     # §11 — homes real axes
RUN_SCAN = False       # §12 — needs od2000/al1342 configured and the laser safe to switch on

ALLOW_MOTION = False   # §5 refuses to enable motion unless this is True

results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(("PASS " if ok else "FAIL ") + name + (f" — {detail}" if detail else ""))

def raises(exc_type, fn, match=None):
    try:
        fn()
    except exc_type as exc:
        return match is None or match in str(exc), f"{type(exc).__name__}: {exc}"
    except Exception as exc:
        return False, f"wrong exception {type(exc).__name__}: {exc}"
    return False, "did not raise"

def pos(axis=None):
    p = dict(zip([a.name for a in lab.gantry._axes], lab.gantry.get_position()))
    return p if axis is None else p[axis]

## 1. Connect in safe_mode

In [ ]:
lab = FlumeLab(CONFIG)
lab.add("gantry")
assert lab.gantry._safe_mode is True, "config should start in safe_mode"
check("connect_all", lab.connect_all())
lab.watch_for_safety()   # ./PAUSE, ./STOP, ./ESTOP sentinel files
st = lab.gantry.get_status()
print(st)
check("starts in safe_mode", st["safe_mode"] is True)
check("starts un-halted", st["halted"] is None)
START = pos()
print("start position:", START)

## 2. Every motion path refuses under safe_mode (no motion)

Each call must raise **before** anything reaches the wire. Positions must not change.

In [ ]:
x = START[TEST_AXIS]
for name, fn in [
    ("move_to", lambda: lab.move_to(**{TEST_AXIS: x + STEP_MM})),
    ("move_to_unfenced", lambda: lab.gantry.move_to_unfenced(TEST_AXIS, x + STEP_MM)),
    ("jog_unfenced", lambda: lab.gantry.jog_unfenced(TEST_AXIS, JOG_SPEED)),
    ("home_axis", lambda: lab.gantry.home_axis(TEST_AXIS)),
    ("locate_limit_switch", lambda: lab.gantry.locate_limit_switch(TEST_AXIS)),
]:
    ok, detail = raises(SnapMotionError, fn, match="safe_mode")
    check(f"safe_mode refuses {name}", ok, detail)

handle = lab.gantry.axis(TEST_AXIS)
check("per-axis handle has no motion methods",
      not any(hasattr(handle, m) for m in ("move_to", "move_by", "begin_move_to", "begin_move_by", "jog")))
time.sleep(0.5)
check("nothing moved", all(abs(pos()[k] - v) < 0.05 for k, v in START.items()), str(pos()))

## 3. Stop/brake commands get through the safe_mode gate (no motion)

`BST`/`ABT`/`STP` and brake-engage (`SOB n 0`) are now allowed under safe_mode, on the PC and on the
Pi agent. Before this change, `pause()` under safe_mode logged "Error soft-stopping …" for every axis.

In [ ]:
class _Capture(logging.Handler):
    def __init__(self): super().__init__(); self.records = []
    def emit(self, r): self.records.append(r)

cap = _Capture(); logging.getLogger("laguna").addHandler(cap)
t0 = time.monotonic()
note = lab.gantry.pause()
dt = time.monotonic() - t0
logging.getLogger("laguna").removeHandler(cap)
errors = [r.getMessage() for r in cap.records if r.levelno >= logging.ERROR]
check("pause() under safe_mode sends BST with no errors", not errors, "; ".join(errors[:3]))
print(f"pause() returned in {dt*1000:.0f} ms (includes the position resync)")
check("gantry latched paused", lab.gantry.halted == "pause")
lab.gantry.resume()
check("resume() clears a pause", lab.gantry.halted is None)

# set_safe_mode(True) now parks the brakes BEFORE closing the gate. Y's brake status is readable (INB 8);
# Z's is not reachable over ASCII — watch Z physically if you want to confirm it.
try:
    check("Y brake engaged while in safe_mode", lab.gantry.brake_is_disengaged("Y") is False)
except Exception as exc:
    check("Y brake engaged while in safe_mode", False, f"could not read: {exc}")

## 4. Fence check (no motion)

Pure Python. The diagonal case is the one the old two-elbow check missed. The fences here are
**virtual** — added for this test and removed straight after.

In [ ]:
from laguna.robot.macron.fences import FenceRegistry, TrajectoryChecker
reg = FenceRegistry(); reg.add(BoxFence("diag_post", 4, 6, 4, 6, 2, 3)); chk = TrajectoryChecker(reg)
check("ribbon catches a fence on the XY+Z diagonal", len(chk.check_ribbon((0, 0, 0), (10, 10, 5))) == 1)
ok, detail = raises(ValueError, lambda: chk.check_point(float("nan"), 0, 0))
check("NaN coordinates refused", ok, detail)
print("configured fences:", [f.describe() for f in lab.gantry.fence_registry.list()])

## 5. Enable motion — explicit

Everything below moves the gantry. Re-read "How to stop motion" at the top.

The next cell is the **STOP cell**. It only pauses; run it any time. Then re-run the cell *after* it to
enable motion.

In [ ]:
# ===================== STOP CELL =====================
lab.pause()          # cancels any move/jog/homing in flight; lab.resume() to continue

In [ ]:
if not ALLOW_MOTION:
    raise SystemExit("ALLOW_MOTION is False — set it True in the first code cell once the rig is clear.")
if lab.safety_state is not SafetyState.RUNNING:
    lab.resume()
check("set_safe_mode(False)", lab.gantry.set_safe_mode(False))
ax = lab.gantry.axis(TEST_AXIS)
print(f"{TEST_AXIS} soft limits after enabling: NLT={ax.get_negative_limit():.1f} PLT={ax.get_positive_limit():.1f}")
X0 = pos(TEST_AXIS)
print(f"{TEST_AXIS} start: {X0:.2f} mm")

## 6. Non-blocking `move_to()`

`move_to()` should return a `MoveHandle` while the axis is still moving. A second move while it runs
must be refused immediately (not queued).

In [ ]:
t0 = time.monotonic()
h = lab.move_to(**{TEST_AXIS: X0 + STEP_MM}, speed=SPEED)
returned_in = time.monotonic() - t0
print(h, f"(returned in {returned_in*1000:.0f} ms)")
check("move_to returns while moving", not h.done)
ok, detail = raises(MotionBusyError, lambda: lab.move_to(**{TEST_AXIS: X0}))
check("second move refused immediately", ok, detail)
ok, detail = raises(TypeError, lambda: bool(h))
check("handle refuses to be a bool", ok)
h.wait()
check("move finished at target", abs(pos(TEST_AXIS) - (X0 + STEP_MM)) < 0.5, f"{pos(TEST_AXIS):.2f}")

## 7. `pause()` cancels a move in flight

A slow move, paused ~1 s in. The axis must stop short, the handle must raise `MotionHalted`, and
motion must be refused until `resume()`.

In [ ]:
target = X0 + 3 * STEP_MM
h = lab.move_to(**{TEST_AXIS: target}, speed=SLOW_SPEED)
time.sleep(1.0)
t0 = time.monotonic()
lab.pause()
print(f"pause() returned in {(time.monotonic()-t0)*1000:.0f} ms")
ok, detail = raises(MotionHalted, lambda: h.wait(timeout=30))
check("paused move raises MotionHalted", ok, detail)
time.sleep(1.0)   # let BST decelerate
stopped_at = pos(TEST_AXIS)
check("axis stopped short of the target", stopped_at < target - 1.0, f"{stopped_at:.2f} vs target {target:.2f}")
ok, detail = raises(MotionHalted, lambda: lab.move_to(**{TEST_AXIS: X0}))
check("motion refused while paused", ok, detail)
lab.resume()
lab.move_to(**{TEST_AXIS: X0}, speed=SPEED).wait()
check("motion allowed again after resume", abs(pos(TEST_AXIS) - X0) < 0.5)

## 8. Jog — and how to stop it

**Ways to stop a jog** (any one is enough):
- `lab.gantry.jog_unfenced(TEST_AXIS, 0)` — the dedicated stop; always allowed, even when halted.
- The **STOP cell** (`lab.pause()`) — decelerates every axis and latches a pause.
- If the cell is still running (`[*]`): **Interrupt the kernel** (⏹ / `I`,`I`). The timed-jog cell
  stops the axis in its `finally`.
- `touch ESTOP` in the repo root, or the physical e-stop.

A jog is open-ended: it's never fence-checked, only stopped by you, a limit, or the controller's soft
limits.

### 8a. Timed jog — stops itself after `JOG_SECONDS`

In [ ]:
j0 = pos(TEST_AXIS)
try:
    lab.gantry.jog_unfenced(TEST_AXIS, JOG_SPEED)
    time.sleep(JOG_SECONDS)
finally:
    lab.gantry.jog_unfenced(TEST_AXIS, 0)   # always runs — including on Interrupt
time.sleep(1.0)
moved = pos(TEST_AXIS) - j0
check("timed jog moved and stopped", abs(moved) > 0.5 * JOG_SPEED * JOG_SECONDS * 0.5, f"moved {moved:.2f} mm")

### 8b. Open-ended jog — returns immediately; **you** stop it

Run the cell, watch the axis, then run **8c** (or the STOP cell) within a few seconds.

In [ ]:
lab.gantry.jog_unfenced(TEST_AXIS, JOG_SPEED)
print(f"{TEST_AXIS} is jogging at {JOG_SPEED} mm/s — run the next cell to stop it.")

In [ ]:
# 8c — STOP THE JOG
lab.gantry.jog_unfenced(TEST_AXIS, 0)
time.sleep(1.0)
a = pos(TEST_AXIS); time.sleep(0.5); b = pos(TEST_AXIS)
check("jog stopped", abs(b - a) < 0.05, f"{a:.2f} -> {b:.2f}")

In [ ]:
# 8d — a jog cannot start while paused, but stopping always works
lab.pause()
ok, detail = raises(MotionHalted, lambda: lab.gantry.jog_unfenced(TEST_AXIS, JOG_SPEED))
check("jog refused while paused", ok, detail)
lab.gantry.jog_unfenced(TEST_AXIS, 0)
check("jog stop allowed while paused", True)
lab.resume()
lab.move_to(**{TEST_AXIS: X0}, speed=SPEED).wait()

## 9. `move_to_unfenced()` passes a fence `move_to()` refuses

A **virtual** fence is placed across `TEST_AXIS`'s path (nothing physical is there), then removed.

In [ ]:
here_pos = pos()
lo = {n: here_pos[n] - 1 for n in ("X", "Y", "Z")}
hi = {n: here_pos[n] + 1 for n in ("X", "Y", "Z")}
lo[TEST_AXIS] = X0 + STEP_MM / 2 - 1; hi[TEST_AXIS] = X0 + STEP_MM / 2 + 1
lab.gantry.fence_registry.add(BoxFence("virtual_test_fence", lo["X"], hi["X"], lo["Y"], hi["Y"], lo["Z"], hi["Z"]))
try:
    ok, detail = raises(FenceViolation, lambda: lab.move_to(**{TEST_AXIS: X0 + STEP_MM}, speed=SPEED))
    check("move_to refuses a path through the fence", ok, detail)
    check("nothing moved", abs(pos(TEST_AXIS) - X0) < 0.05)
    lab.gantry.move_to_unfenced(TEST_AXIS, X0 + STEP_MM, speed=SPEED).wait()
    check("move_to_unfenced crosses it", abs(pos(TEST_AXIS) - (X0 + STEP_MM)) < 0.5)
finally:
    lab.gantry.fence_registry.remove("virtual_test_fence")
lab.move_to(**{TEST_AXIS: X0}, speed=SPEED).wait()

## 10. `estop()` → motion refused → `rearm()` stays in safe_mode

After the estop the motors are off and Y/Z brakes are engaged. Before this change the next `move_to()`
drove straight into them, and `rearm()` silently re-enabled motion.

In [ ]:
h = lab.move_to(**{TEST_AXIS: X0 + 2 * STEP_MM}, speed=SLOW_SPEED)
time.sleep(1.0)
t0 = time.monotonic()
lab.estop()
print(f"estop() returned in {(time.monotonic()-t0)*1000:.0f} ms")
ok, detail = raises(MotionHalted, lambda: h.wait(timeout=30))
check("estop cancels the move", ok, detail)
check("rig ESTOPPED", lab.safety_state is SafetyState.ESTOPPED)
ok, detail = raises(MotionHalted, lambda: lab.move_to(**{TEST_AXIS: X0}))
check("motion refused after estop", ok, detail)
ok, detail = raises(RuntimeError, lambda: lab.resume(), match="rearm")
check("resume() refused from estop", ok, detail)

check("rearm()", lab.rearm())
check("rearm leaves safe_mode ON", lab.gantry._safe_mode is True)
ok, detail = raises(SnapMotionError, lambda: lab.move_to(**{TEST_AXIS: X0}), match="safe_mode")
check("motion still refused until set_safe_mode(False)", ok, detail)
check("set_safe_mode(False) re-enables", lab.gantry.set_safe_mode(False))
lab.resume()   # restarts the clock; no schedule running here
lab.move_to(**{TEST_AXIS: X0}, speed=SPEED).wait()

## 11. Homing *(optional — `RUN_HOMING = True`)*

Prints each axis's travel-based search timeout, homes `TEST_AXIS` non-blocking, then starts a second
homing and pauses it. **Homing ignores fences** — make sure the path to the home switch is clear.

In [ ]:
if RUN_HOMING:
    cfg = lab.gantry.homing._config
    for axis, acfg in cfg.axis_configs.items():
        print(f"{axis.name}: max_travel_mm={acfg.max_travel_mm} -> search timeout {cfg.search_timeout_s(acfg):.0f} s")
    h = lab.gantry.home_axis(TEST_AXIS)
    check("home_axis returns a handle while moving", not h.done)
    standoff = h.wait().result
    check("home_axis finished", standoff is not None, f"standoff {standoff}")

    h = lab.gantry.home_axis(TEST_AXIS)
    time.sleep(2.0)
    lab.pause()
    ok, detail = raises(MotionHalted, lambda: h.wait(timeout=60))
    check("pause cancels homing", ok, detail)
    lab.resume()
else:
    print("skipped (RUN_HOMING = False)")

## 12. Pi-agent rangefinder scan *(optional — `RUN_SCAN = True`)*

Starts a pass along `TEST_AXIS` and pauses it partway. Expect `MotionHalted` mentioning a **partial
profile kept at …**, and that CSV on disk. Then a full pass should complete normally.

In [ ]:
if RUN_SCAN:
    names = [a.name for a in lab.gantry._axes]
    start = lab.gantry.get_position()
    end = list(start); end[names.index(TEST_AXIS)] = start[names.index(TEST_AXIS)] + 3 * STEP_MM
    import threading
    threading.Timer(2.0, lab.pause).start()
    ok, detail = raises(MotionHalted, lambda: lab.acquire_scan("od2000", end=end, feed_rate_mm_s=SLOW_SPEED, axis=TEST_AXIS))
    check("paused scan raises and keeps the partial profile", ok and "partial profile kept" in detail, detail)
    lab.resume()
    lab.move_to(start).wait()
    result = lab.acquire_scan("od2000", end=end, feed_rate_mm_s=SPEED, axis=TEST_AXIS)
    check("full scan completes", result.path.exists(), f"{result.path} ({result.metadata.get('samples')} samples)")
    lab.move_to(start).wait()
else:
    print("skipped (RUN_SCAN = False)")

## 13. Summary, park, safe_mode back on

In [ ]:
try:
    lab.move_to(**{TEST_AXIS: START[TEST_AXIS]}, speed=SPEED).wait()
except Exception as exc:
    print("could not return to start:", exc)
lab.gantry.set_safe_mode(True)
print(f"\n{sum(ok for _, ok, _ in results)}/{len(results)} checks passed")
for name, ok, detail in results:
    if not ok:
        print("FAIL", name, "—", detail)

In [ ]:
lab.disconnect_all()